# TRACK-FA Progression Deep Learning


## 1. Seeds and Reproducibility

This cell fixes random seeds before any stochastic split or model initialisation.


In [14]:
# Seed numpy, random, and torch before creating any stochastic split or model.
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


REPO_ROOT = find_project_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds

set_global_seeds(42)


## 2. Imports, Hardware, and Configuration

This cell loads analysis and training utilities, chooses the best available compute backend, and sets runtime controls. Keep clinical-head weights at zero unless the run is explicitly clinical-target supervised.


In [15]:
# Import analysis and modeling tools; then configure training settings.
import os
import time
import warnings

import numpy as np
import pandas as pd
try:
    import torch
    TORCH_AVAILABLE = True
except ModuleNotFoundError:
    torch = None
    TORCH_AVAILABLE = False
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from src.data.audit import modelling_pair_count_table
from src.data.qc import filter_complete_pairs
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import group_kfold_indices, subject_loo_split
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import clinical_change_effect_sizes, paired_cohens_d, reference_effect_sizes, srm, paired_ttest, rmse
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary
if TORCH_AVAILABLE:
    from src.eval.shap import _pick_best_combo, run_shap_on_combo, TOP_N_FEATURES
else:
    _pick_best_combo = run_shap_on_combo = None
    TOP_N_FEATURES = 12
if TORCH_AVAILABLE:
    from src.training.fusion import (
        evaluate_fusion_loss,
        prepare_fusion_arrays,
        train_fusion_model,
    )
    from src.training.pair import prepare_pair_arrays, train_pair_model
else:
    evaluate_fusion_loss = prepare_fusion_arrays = train_fusion_model = None
    prepare_pair_arrays = train_pair_model = None

warnings.filterwarnings('ignore')

config = DEFAULT_CONFIG
RANDOM_SEED = config.random_state
CV_N_SPLITS = config.cv_n_splits
N_BOOT = 300
FUSION_CV_MODE = "group_kfold"
PAIR_CV_MODE = "group_kfold"

if TORCH_AVAILABLE and torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif TORCH_AVAILABLE and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif TORCH_AVAILABLE:
    DEVICE = torch.device('cpu')
else:
    DEVICE = 'torch-unavailable'

N_THREADS = max(1, min(os.cpu_count() or 1, 10))
if TORCH_AVAILABLE:
    torch.set_num_threads(N_THREADS)
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

# CV defaults: both DL architectures use grouped participant folds so the notebook finishes reliably.
# Strict LOO can be enabled after a grouped-fold candidate is worth confirming.
# Clinical heads remain disabled. DL regularisation is tuned through a compact
# grid of learning rate, weight decay, dropout, epoch budget, and patience;
# strict LOO is reserved for the selected PairModel setting so the notebook can finish.
VAL_FRACTION = 0.2
USE_CLINICAL_HEADS = False
LAMBDA_FARS = 0.0
LAMBDA_SARA = 0.0
LAMBDA_PROG = 1.0
RUN_SHAP = False

DL_REGULARIZATION_GRID = [
    {"name": "low_decay_no_dropout", "lr": 3e-3, "weight_decay": 1e-5, "dropout": 0.0, "z_clip": None},
    {"name": "low_decay_clip3", "lr": 3e-3, "weight_decay": 1e-5, "dropout": 0.0, "z_clip": 3.0},
]

FUSION_TRAIN_KWARGS = dict(
    epochs=60,
    patience=8,
    lr=3e-3,
    weight_decay=1e-5,
    dropout=0.0,
    z_clip=3.0,
    val_fraction=VAL_FRACTION,
    seed=RANDOM_SEED,
    use_clinical_heads=USE_CLINICAL_HEADS,
    lambda_prog=LAMBDA_PROG,
    lambda_fars=LAMBDA_FARS,
    lambda_sara=LAMBDA_SARA,
)
PAIR_TRAIN_KWARGS = dict(
    epochs=60,
    patience=8,
    lr=3e-3,
    weight_decay=1e-5,
    dropout=0.0,
    z_clip=None,
    val_fraction=VAL_FRACTION,
    seed=RANDOM_SEED,
    use_clinical_heads=USE_CLINICAL_HEADS,
    lambda_prog=LAMBDA_PROG,
    lambda_fars=LAMBDA_FARS,
    lambda_sara=LAMBDA_SARA,
)

DATA_PATH = config.processed_data_dir / 'trackfa_pairs_drop3poms.csv'
print('Working directory:', config.repo_root)
print('Data path:', DATA_PATH, 'OK' if DATA_PATH.exists() else 'MISSING')
print('Torch device:', DEVICE, '| CPU threads:', N_THREADS, '| grouped CV folds:', CV_N_SPLITS)
print({'fusion_cv_mode': FUSION_CV_MODE, 'pair_cv_mode': PAIR_CV_MODE})
print('Fusion training settings:', FUSION_TRAIN_KWARGS)
print('Pair training settings:', PAIR_TRAIN_KWARGS)
print('DL regularisation/tuning grid:', DL_REGULARIZATION_GRID)


Working directory: /Users/robertwang/Documents/New_project/biomarkers
Data path: /Users/robertwang/Documents/New_project/biomarkers/data/processed/trackfa_pairs_drop3poms.csv OK
Torch device: mps | CPU threads: 10 | grouped CV folds: 5
{'fusion_cv_mode': 'group_kfold', 'pair_cv_mode': 'group_kfold'}
Fusion training settings: {'epochs': 60, 'patience': 8, 'lr': 0.003, 'weight_decay': 1e-05, 'dropout': 0.0, 'z_clip': 3.0, 'val_fraction': 0.2, 'seed': 42, 'use_clinical_heads': False, 'lambda_prog': 1.0, 'lambda_fars': 0.0, 'lambda_sara': 0.0}
Pair training settings: {'epochs': 60, 'patience': 8, 'lr': 0.003, 'weight_decay': 1e-05, 'dropout': 0.0, 'z_clip': None, 'val_fraction': 0.2, 'seed': 42, 'use_clinical_heads': False, 'lambda_prog': 1.0, 'lambda_fars': 0.0, 'lambda_sara': 0.0}
DL regularisation/tuning grid: [{'name': 'low_decay_no_dropout', 'lr': 0.003, 'weight_decay': 1e-05, 'dropout': 0.0, 'z_clip': None}, {'name': 'low_decay_clip3', 'lr': 0.003, 'weight_decay': 1e-05, 'dropout': 0

## 3. Load Paired TRACK-FA Data

The paired table is converted into visit-level rows for model input, while progression intervals remain available for held-out score differencing.


In [16]:
# Load the paired TRACK-FA table and convert it to visit-level rows.
pairs_df = pd.read_csv(DATA_PATH)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
groups = infer_trackfa_feature_groups(pairs_df)
long_df = trackfa_pairs_to_long(pairs_df)
subject_col = 'pair_id'  # progression interval id, e.g. AAN001_V1V2
split_group_col = 'subject'  # participant id; keeps V1V2 and V2V3 in the same fold

structural_cols = list(groups.poms + groups.brainspinemorph)
diffusion_cols = list(groups.braindti)
imaging_cols = sorted({c for c in structural_cols + diffusion_cols if c in long_df.columns})
structural_cols = [c for c in structural_cols if c in imaging_cols]
diffusion_cols = [c for c in diffusion_cols if c in imaging_cols]

print('Pairs:', pairs_df.shape)
print('Long rows:', len(long_df), 'pairs:', long_df[subject_col].nunique())
print('Imaging feature counts:', {
    'all_imaging': len(imaging_cols),
    'structural': len(structural_cols),
    'diffusion': len(diffusion_cols),
})


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Pairs: (207, 455)
Long rows: 414 pairs: 207
Imaging feature counts: {'all_imaging': 146, 'structural': 42, 'diffusion': 108}


## 4. Define Training Helper

The helper trains each exploratory architecture on all imaging features. It keeps the same leakage rule as the classical models: split by participant group, not individual interval alone.


In [17]:
# Train one leakage-safe progression model per architecture using all imaging features.
model_spec = {'name': 'all_imaging', 'features': imaging_cols}
combinations = [model_spec]

combo_meta = {
    'all_imaging': {
        'features': imaging_cols,
        'struct_idx': [i for i, f in enumerate(imaging_cols) if f in structural_cols],
        'diff_idx': [i for i, f in enumerate(imaging_cols) if f in diffusion_cols],
        'back_idx': [],
    }
}

print('Model feature pool:', model_spec['name'], len(imaging_cols))
n_pairs = long_df[split_group_col].nunique()
fusion_fits = CV_N_SPLITS if FUSION_CV_MODE == 'group_kfold' else n_pairs
pair_fits = CV_N_SPLITS if PAIR_CV_MODE == 'group_kfold' else n_pairs
print('Fusion model fits:', fusion_fits)
print('Pair model fits:', pair_fits)


Model feature pool: all_imaging 146
Fusion model fits: 5
Pair model fits: 5


## 5. Build Subject-Level Splits

This function creates grouped train/test splits. If a participant contributes both `V1V2` and `V2V3`, both intervals move together into either training or testing.


In [18]:
def make_subject_splits(groups_arr, *, mode, n_splits, seed):
    if mode == 'loo':
        return list(subject_loo_split(groups_arr))
    if mode == 'group_kfold':
        return list(group_kfold_indices(groups_arr, n_splits=n_splits, seed=seed))
    raise ValueError(f'Unknown CV mode: {mode}')


def paired_delta_array(oof_df):
    if oof_df.empty:
        return np.asarray([], dtype=float)
    return (
        oof_df.sort_values(['subject', 'visit'])
        .groupby('subject')['score']
        .apply(lambda x: list(x)[1] - list(x)[0] if len(x) == 2 else np.nan)
        .dropna()
        .to_numpy(dtype=float)
    )


## 6. FusionMLP Grouped-CV Run

FusionMLP is kept in grouped-fold mode for runtime control. Its result should be interpreted as exploratory evidence rather than the primary progression biomarker.


In [19]:
if not TORCH_AVAILABLE:
    print("PyTorch is not installed in this kernel; skipping DL model training while keeping the notebook executable.")
    fusion_results = []
    fusion_models = {}
    fusion_results_df = pd.DataFrame()
else:
    # FusionMLP grouped-CV: kept fast because this architecture is exploratory and weaker in smoke runs.
    fusion_results = []
    fusion_models = {}

    for combo in combinations:
        feats = combo['features']
        meta = combo_meta[combo['name']]
        sub = filter_complete_pairs(long_df, subject_col, feats).reset_index(drop=True)
        groups_arr = sub[split_group_col].values
        if sub[split_group_col].nunique() < CV_N_SPLITS:
            print('Skipping FusionMLP', combo['name'], 'not enough complete pairs')
            continue

        oof_rows = []
        epochs_used = []
        start_time = time.time()
        splits = make_subject_splits(groups_arr, mode=FUSION_CV_MODE, n_splits=CV_N_SPLITS, seed=RANDOM_SEED)
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            train = sub.iloc[train_idx].copy()
            test = sub.iloc[test_idx].copy()
            scaler = StandardScaler().fit(train[feats].values)
            model, best_epoch = train_fusion_model(
                train,
                feats,
                meta,
                scaler,
                subject_col=subject_col,
                split_group_col=split_group_col,
                device=DEVICE,
                **FUSION_TRAIN_KWARGS,
            )
            epochs_used.append(best_epoch)
            model.eval()
            test_arrays = prepare_fusion_arrays(
                test,
                feats,
                meta,
                scaler,
                subject_col=subject_col,
                device=DEVICE,
                z_clip=FUSION_TRAIN_KWARGS.get("z_clip"),
            )
            with torch.inference_mode():
                _, _, _, prog_out = evaluate_fusion_loss(
                    model,
                    test_arrays,
                    use_clinical_heads=USE_CLINICAL_HEADS,
                    lambda_prog=LAMBDA_PROG,
                    lambda_fars=LAMBDA_FARS,
                    lambda_sara=LAMBDA_SARA,
                )
            scores = prog_out.detach().cpu().numpy().ravel()
            fold_rows = test[[subject_col, 'visit']].copy()
            fold_rows['score'] = scores
            fold_rows['method'] = f'FusionMLP_{FUSION_CV_MODE}'
            fold_rows['fold'] = fold
            oof_rows.append(fold_rows.rename(columns={subject_col: 'subject'}))

        oof_df = pd.concat(oof_rows, ignore_index=True) if oof_rows else pd.DataFrame()
        d, mean_diff, sd_diff, n = paired_cohens_d(oof_df, 'subject')
        paired_deltas = paired_delta_array(oof_df)
        interval_summary = adjacent_pair_interval_effect_summary(
            oof_df,
            pair_col="subject",
            visit_col="visit",
            score_col="score",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        )
        annual_diag = annual_tuning_diagnostics(interval_summary)
        runtime = time.time() - start_time
        row = {
            'model': f'FusionMLP_{FUSION_CV_MODE}',
            'method': FUSION_CV_MODE,
            'feature_pool': combo['name'],
            'n_features': len(feats),
            'n_subjects': n,
            'd': d,
            'd_score': d,
            'srm': srm(oof_df, 'subject'),
            **annual_diag,
            'mean_diff': mean_diff,
            'sd_diff': sd_diff,
            'p_value': paired_ttest(oof_df, 'subject'),
            'delta_min': float(paired_deltas.min()) if len(paired_deltas) else np.nan,
            'delta_max': float(paired_deltas.max()) if len(paired_deltas) else np.nan,
            'delta_abs_mean': float(np.mean(np.abs(paired_deltas))) if len(paired_deltas) else np.nan,
            'mean_best_epoch': float(np.mean(epochs_used)) if epochs_used else np.nan,
            'runtime_sec': runtime,
            'oof_df': oof_df,
        }
        fusion_results.append(row)
        fusion_models[combo['name']] = {'last_model': model, 'last_scaler': scaler, 'features': feats, 'meta': meta}
        print(f"FusionMLP {combo['name']}: d={d:.3f}, n={n}, mode={FUSION_CV_MODE}, fits={len(splits)}, {runtime:.1f}s")

    fusion_results_df = pd.DataFrame(fusion_results).drop(columns=['oof_df'], errors='ignore')
    display(fusion_results_df.sort_values('d', ascending=False))


FusionMLP all_imaging: d=0.134, n=207, mode=group_kfold, fits=5, 36.6s


,model,method,feature_pool,n_features,n_subjects,d,d_score,srm,dz_v1_v2,dz_v2_v3,...,annual_interval_gap,p_progression,mean_diff,sd_diff,p_value,delta_min,delta_max,delta_abs_mean,mean_best_epoch,runtime_sec
0,FusionMLP_group_kfold,group_kfold,all_imaging,146,207,0.133952,0.133952,0.133952,0.026331,0.261363,...,0.235032,0.56271,0.000529,0.003949,0.055327,-0.011009,0.012929,0.002953,29.2,36.568463


## 7. PairModel Run

PairModel directly consumes paired visits and predicts a progression score. Use participant-level LOO only when runtime is acceptable; grouped folds are the practical fallback.


In [20]:
if not TORCH_AVAILABLE:
    print("PyTorch is not installed in this kernel; skipping DL model training while keeping the notebook executable.")
    pair_results = []
    pair_models = {}
    pair_results_df = pd.DataFrame()
else:
    # PairModel subject-level LOO: final exploratory DL result, still held out by pair_id.
    pair_results = []
    pair_models = {}

    for combo in combinations:
        feats = combo['features']
        sub = filter_complete_pairs(long_df, subject_col, feats).reset_index(drop=True)
        groups_arr = sub[split_group_col].values
        if sub[split_group_col].nunique() < CV_N_SPLITS:
            print('Skipping PairModel', combo['name'], 'not enough complete pairs')
            continue

        oof_rows = []
        epochs_used = []
        start_time = time.time()
        splits = make_subject_splits(groups_arr, mode=PAIR_CV_MODE, n_splits=CV_N_SPLITS, seed=RANDOM_SEED)
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            train = sub.iloc[train_idx].copy()
            test = sub.iloc[test_idx].copy()
            scaler = StandardScaler().fit(train[feats].values)
            fold_kwargs = dict(PAIR_TRAIN_KWARGS, seed=RANDOM_SEED + fold)
            model, best_epoch = train_pair_model(
                train,
                feats,
                scaler,
                subject_col=subject_col,
                split_group_col=split_group_col,
                device=DEVICE,
                **fold_kwargs,
            )
            epochs_used.append(best_epoch)
            model.eval()
            X1, X2, sids, *_ = prepare_pair_arrays(
                test,
                feats,
                scaler,
                subject_col=subject_col,
                device=DEVICE,
                z_clip=PAIR_TRAIN_KWARGS.get("z_clip"),
            )
            with torch.inference_mode():
                prog, _, _, _, _ = model(X1, X2)
            deltas = prog.detach().cpu().numpy().ravel()
            for sid, delta_score in zip(sids, deltas):
                oof_rows.append({'subject': sid, 'visit': 1, 'score': -0.5 * float(delta_score), 'method': f'PairModel_{PAIR_CV_MODE}', 'fold': fold})
                oof_rows.append({'subject': sid, 'visit': 2, 'score': 0.5 * float(delta_score), 'method': f'PairModel_{PAIR_CV_MODE}', 'fold': fold})

        oof_df = pd.DataFrame(oof_rows)
        d, mean_diff, sd_diff, n = paired_cohens_d(oof_df, 'subject')
        paired_deltas = paired_delta_array(oof_df)
        interval_summary = adjacent_pair_interval_effect_summary(
            oof_df,
            pair_col="subject",
            visit_col="visit",
            score_col="score",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        )
        annual_diag = annual_tuning_diagnostics(interval_summary)
        runtime = time.time() - start_time
        row = {
            'model': f'PairModel_{PAIR_CV_MODE}',
            'method': PAIR_CV_MODE,
            'feature_pool': combo['name'],
            'n_features': len(feats),
            'n_subjects': n,
            'd': d,
            'd_score': d,
            'srm': srm(oof_df, 'subject'),
            **annual_diag,
            'mean_diff': mean_diff,
            'sd_diff': sd_diff,
            'p_value': paired_ttest(oof_df, 'subject'),
            'delta_min': float(paired_deltas.min()) if len(paired_deltas) else np.nan,
            'delta_max': float(paired_deltas.max()) if len(paired_deltas) else np.nan,
            'delta_abs_mean': float(np.mean(np.abs(paired_deltas))) if len(paired_deltas) else np.nan,
            'mean_best_epoch': float(np.mean(epochs_used)) if epochs_used else np.nan,
            'runtime_sec': runtime,
            'oof_df': oof_df,
        }
        pair_results.append(row)
        pair_models[combo['name']] = {'last_model': model, 'last_scaler': scaler, 'features': feats, 'meta': combo_meta[combo['name']]}
        print(f"PairModel {combo['name']}: d={d:.3f}, n={n}, mode={PAIR_CV_MODE}, fits={len(splits)}, {runtime:.1f}s")

    pair_results_df = pd.DataFrame(pair_results).drop(columns=['oof_df'], errors='ignore')
    display(pair_results_df.sort_values('d', ascending=False))


PairModel all_imaging: d=0.368, n=207, mode=group_kfold, fits=5, 3.5s


,model,method,feature_pool,n_features,n_subjects,d,d_score,srm,dz_v1_v2,dz_v2_v3,...,annual_interval_gap,p_progression,mean_diff,sd_diff,p_value,delta_min,delta_max,delta_abs_mean,mean_best_epoch,runtime_sec
0,PairModel_group_kfold,group_kfold,all_imaging,146,207,0.367662,0.367662,0.367662,0.534474,0.193064,...,0.341409,0.690236,0.023218,0.06315,3.119569e-07,-0.208019,0.257724,0.04729,22.2,3.475045


## 8. Aggregate DL Results

Out-of-fold progression scores are differenced by interval and summarised with paired Cohen's `d_z`.


In [21]:
# Aggregate paired Cohen's d results and display the model comparison table.
summary_parts = [df for df in [fusion_results_df, pair_results_df] if isinstance(df, pd.DataFrame) and not df.empty]
summary_df = (
    pd.concat(summary_parts, ignore_index=True)
      .sort_values(['mean_validation_annual_dz', 'annual_interval_gap'], ascending=[False, True])
      .reset_index(drop=True)
) if summary_parts else pd.DataFrame()

dl_optimization_rows = []
for row in fusion_results + pair_results:
    train_params = FUSION_TRAIN_KWARGS if str(row.get('model', '')).startswith('FusionMLP') else PAIR_TRAIN_KWARGS
    dl_optimization_rows.append(optimization_row(
        model=row.get('model', 'DL'),
        params={
            'feature_pool': row.get('feature_pool', 'all_imaging'),
            'cv_mode': row.get('method'),
            **{k: train_params[k] for k in ['epochs', 'patience', 'lr', 'weight_decay', 'dropout', 'z_clip'] if k in train_params},
        },
        result=row,
        runtime_sec=row.get('runtime_sec', np.nan),
        notes='clinical heads disabled; tuned/reported on mean annual d_z from V1->V2 and V2->V3 with interval-gap diagnostic',
    ))
dl_optimization_df = optimization_log(dl_optimization_rows, sort_by="mean_validation_annual_dz")
log_path = save_optimization_log(dl_optimization_df, REPO_ROOT / 'results' / 'progression_dl_optimization_log.csv')
print('Saved optimization log:', log_path)
display(summary_df.groupby("model", as_index=False, sort=False).head(1) if not summary_df.empty else summary_df)
print("DL tuning candidates evaluated:", len(dl_optimization_df))

for review_model, review_group in dl_optimization_df.groupby("model", sort=False) if not dl_optimization_df.empty else []:
    dl_review = tuning_recommendation(review_group)
    print(f"Tuning review: {review_model}")
    print("Numerically best configuration")
    display(pd.DataFrame([dl_review["raw_best"]]))
    print("One-SE / near-optimal candidate count:", len(dl_review["near_optimal"]))
    print("Recommended configuration by implemented hierarchy")
    display(pd.DataFrame([dl_review["recommended"]]))
    print(dl_review["summary"])
    print("Human-verification summary")
    display(tuning_verification_summary(dl_review))


Saved optimization log: /Users/robertwang/Documents/New_project/biomarkers/results/progression_dl_optimization_log.csv


,model,method,feature_pool,n_features,n_subjects,d,d_score,srm,dz_v1_v2,dz_v2_v3,...,annual_interval_gap,p_progression,mean_diff,sd_diff,p_value,delta_min,delta_max,delta_abs_mean,mean_best_epoch,runtime_sec
0,PairModel_group_kfold,group_kfold,all_imaging,146,207,0.367662,0.367662,0.367662,0.534474,0.193064,...,0.341409,0.690236,0.023218,0.063150,3.119569e-07,-0.208019,0.257724,0.047290,22.2,3.475045
1,FusionMLP_group_kfold,group_kfold,all_imaging,146,207,0.133952,0.133952,0.133952,0.026331,0.261363,...,0.235032,0.562710,0.000529,0.003949,5.532673e-02,-0.011009,0.012929,0.002953,29.2,36.568463


DL tuning candidates evaluated: 2
Tuning review: PairModel_group_kfold
Numerically best configuration


,param_feature_pool,param_cv_mode,param_epochs,param_patience,param_lr,param_weight_decay,param_dropout,param_z_clip,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,all_imaging,group_kfold,60,8,0.003,0.00001,0.0,NaN,0.534474,0.193064,0.363769,0.341409,0.690236,NaN,NaN,NaN,NaN,NaN,1.0


One-SE / near-optimal candidate count: 1
Recommended configuration by implemented hierarchy


...

,param_feature_pool,param_cv_mode,param_epochs,param_patience,param_lr,param_weight_decay,param_dropout,param_z_clip,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
1,all_imaging,group_kfold,60,8,0.003,0.00001,0.0,3.0,0.026331,0.261363,0.143847,0.235032,0.56271,NaN,NaN,NaN,NaN,NaN,1.0


One-SE / near-optimal candidate count: 1
Recommended configuration by implemented hierarchy


,param_feature_pool,param_cv_mode,param_epochs,param_patience,param_lr,param_weight_decay,param_dropout,param_z_clip,dz_v1_v2,dz_v2_v3,...,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank,directional_consistency,score_ranking_stability
1,all_imaging,group_kfold,60,8,0.003,0.00001,0.0,3.0,0.026331,0.261363,...,0.235032,0.56271,inf,-inf,-inf,-inf,NaN,1.0,-inf,-inf


Recommended candidate is also the raw best by mean annual validation d_z; the consistency, progression-probability, simplicity, and stability tie-breakers did not select a different row.
Human-verification summary


,item,value
0,Best raw-performance parameters,"{'feature_pool': 'all_imaging', 'cv_mode': 'gr..."
1,Recommended parameters,"{'feature_pool': 'all_imaging', 'cv_mode': 'gr..."
2,Difference in performance,0.0
3,Reason for recommendation,Recommended candidate is also the raw best by ...
4,Any instability/warning,No automatic warning.


## 9. Optional SHAP Setup

SHAP is optional because it can be expensive. It should use the same all-imaging run that produced the reported model result.


In [22]:
# Keep optional SHAP on the same all-imaging run used for model reporting.
best_fusion_combo = 'all_imaging' if len(fusion_results_df) else None
best_pair_combo = 'all_imaging' if len(pair_results_df) else None
print('Fusion SHAP target:', best_fusion_combo)
print('Pair SHAP target:', best_pair_combo)


Fusion SHAP target: all_imaging
Pair SHAP target: all_imaging


## 10. Optional FusionMLP Explanation

This section explains the FusionMLP result only if SHAP is enabled. Skipping it does not affect the progression benchmark table.


In [23]:
if RUN_SHAP and best_fusion_combo is not None:
    # Explain the FusionMLP progression head with SHAP values, table-only.
    fusion_shap = run_shap_on_combo(
        model_kind='fusion',
        combo_name=best_fusion_combo,
        combinations=combinations,
        combo_meta=combo_meta,
        long_df=long_df,
        subject_col=subject_col,
        device=DEVICE,
        seed=RANDOM_SEED,
        train_kwargs=FUSION_TRAIN_KWARGS,
    )
    print('FusionMLP modality summary')
    display(fusion_shap['modality_summary'])
else:
    print('Skipping FusionMLP SHAP because RUN_SHAP=False.')


Skipping FusionMLP SHAP because RUN_SHAP=False.


## 11. Optional PairModel Explanation

This section explains the PairModel result only if SHAP is enabled. Keep interpretation cautious because DL results remain exploratory.


In [24]:
if RUN_SHAP and best_pair_combo is not None:
    # Explain the PairModel progression head with SHAP values, table-only.
    pair_shap = run_shap_on_combo(
        model_kind='pair',
        combo_name=best_pair_combo,
        combinations=combinations,
        combo_meta=combo_meta,
        long_df=long_df,
        subject_col=subject_col,
        device=DEVICE,
        seed=RANDOM_SEED,
        train_kwargs=PAIR_TRAIN_KWARGS,
    )
    print('PairModel modality summary')
    display(pair_shap['modality_summary'])
else:
    print('Skipping PairModel SHAP because RUN_SHAP=False.')


Skipping PairModel SHAP because RUN_SHAP=False.


## 12. Summary Outputs

This cell prints the number of collected summary rows and confirms that the notebook produced comparable progression outputs.


In [25]:
print('Summary rows:', len(summary_df))
print('FusionMLP SHAP target:', best_fusion_combo)
print('PairModel SHAP target:', best_pair_combo)
summary_df.sort_values('d', ascending=False) if 'd' in summary_df.columns else summary_df


Summary rows: 2
FusionMLP SHAP target: all_imaging
PairModel SHAP target: all_imaging


,model,method,feature_pool,n_features,n_subjects,d,d_score,srm,dz_v1_v2,dz_v2_v3,...,annual_interval_gap,p_progression,mean_diff,sd_diff,p_value,delta_min,delta_max,delta_abs_mean,mean_best_epoch,runtime_sec
0,PairModel_group_kfold,group_kfold,all_imaging,146,207,0.367662,0.367662,0.367662,0.534474,0.193064,...,0.341409,0.690236,0.023218,0.063150,3.119569e-07,-0.208019,0.257724,0.047290,22.2,3.475045
1,FusionMLP_group_kfold,group_kfold,all_imaging,146,207,0.133952,0.133952,0.133952,0.026331,0.261363,...,0.235032,0.562710,0.000529,0.003949,5.532673e-02,-0.011009,0.012929,0.002953,29.2,36.568463


## 13. Clinical Benchmark Table

The final table compares the best exploratory DL progression score with FARS, SARA, and the strongest single imaging feature. Clinical benchmarks use adjacent visit changes only.


In [26]:
# Clinical benchmark comparison for the best exploratory progression score.
from src.eval.metrics import bootstrap_ci_d

imaging_ref = reference_effect_sizes(long_df, imaging_cols, scale_cols=(), subject_col=subject_col, visit_col="visit")
clinical_ref = clinical_change_effect_sizes(pairs_df, scale_cols=("FARS", "SARA"), pair_types=("V1V2", "V2V3"))

all_dl = pd.concat([pd.DataFrame(fusion_results), pd.DataFrame(pair_results)], ignore_index=True)
if len(all_dl):
    best_dl = all_dl.sort_values(["mean_validation_annual_dz", "annual_interval_gap"], ascending=[False, True]).head(1).iloc[0]
    best_oof = best_dl.get("oof_df") if "oof_df" in best_dl.index else None
    ci_low = ci_high = np.nan
    if isinstance(best_oof, pd.DataFrame) and len(best_oof):
        _, ci_low, ci_high = bootstrap_ci_d(best_oof.rename(columns={"score": "value"}), "subject", "visit", "value")
    model_row = pd.DataFrame([{
        "feature": f"{best_dl.get('model', 'DL progression')} / {best_dl.get('feature_pool', '')}",
        "kind": "model",
        "d": best_dl.get("d_score", np.nan),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "source_delta_col": np.nan,
        "pair_types": np.nan,
    }])
else:
    model_row = pd.DataFrame([{"feature": "DL progression", "kind": "model", "d": np.nan, "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": np.nan, "pair_types": np.nan}])

bench_rows = []
for scale in ("FARS", "SARA"):
    hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
    if len(hit):
        bench_rows.append(hit.iloc[0].to_dict())
top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
if len(top_img):
    bench_rows.append(top_img.iloc[0].to_dict())
display(pd.concat([model_row, pd.DataFrame(bench_rows)], ignore_index=True))


,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types,mean_diff,sd_diff,n_pairs
0,PairModel_group_kfold / all_imaging,model,0.367662,0.235969,0.514256,NaN,NaN,NaN,NaN,NaN
1,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3",2.001610,4.912805,207.0
2,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3",1.070048,2.639077,207.0
3,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN,-1615.648449,2419.287337,207.0
